# Actividad 15v3 — XGBoost como Modelo Competidor
**Autor:** Fabrizio Sanchez Saravia — UPeU Juliaca

## Justificacion
XGBoost es el competidor mas peligroso para LSTM-Attention en Small Data porque:
- Acepta las mismas 23 variables exogenas (clima NASA, emergencias INDECI)
- No sobreajusta con n=44 si se regulariza correctamente
- En competencias de series temporales con pocos datos frecuentemente supera a LSTM
- Permite calcular feature importance para comparar con SHAP del GE

## Hipotesis
Si XGBoost supera al GM v2 en MAE global pero no en meses de shock,
confirma que la arquitectura LSTM-Attention tiene ventaja estructural en volatilidad.

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path('../..')
DATA_PATH    = PROJECT_ROOT / 'data/processed/master_dataset_fase2_multivariado.csv'
NLP_PATH     = PROJECT_ROOT / 'notebooks/fase2/output/01_nlp_sentimiento/sentimiento_mensual.csv'
GE_METRICAS  = PROJECT_ROOT / 'resultados/ge/ge_metricas.json'
OUT_DIR      = PROJECT_ROOT / 'resultados/xgboost'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('XGBoost version:', xgb.__version__)
print('DATA_PATH ok:', DATA_PATH.exists())
print('NLP_PATH  ok:', NLP_PATH.exists())

In [ ]:
# Carga y agregacion por media provincial (igual que GE y GM v2)
df_raw = pd.read_csv(DATA_PATH, parse_dates=['fecha_evento'])
df = df_raw.groupby('fecha_evento').mean(numeric_only=True).reset_index()
df = df.sort_values('fecha_evento').reset_index(drop=True)
print(f'Agregado: {df.shape} | {df["fecha_evento"].min().date()} -> {df["fecha_evento"].max().date()}')

# NLP
df_nlp = pd.read_csv(NLP_PATH, encoding='utf-8-sig')
fc = [c for c in df_nlp.columns if any(k in c.lower() for k in ['fecha','periodo','mes','month'])][0]
df_nlp = df_nlp.rename(columns={fc: 'fecha_evento'})
df_nlp['fecha_evento'] = pd.to_datetime(df_nlp['fecha_evento'])
df_nlp['nlp_index']      = df_nlp['avg_sentiment'] * np.log1p(df_nlp['n_noticias_beto'])
df_nlp['nlp_index_lag1'] = df_nlp['nlp_index'].shift(1).fillna(0)

df = df.merge(df_nlp[['fecha_evento','nlp_index','nlp_index_lag1']], on='fecha_evento', how='left')
df['nlp_index']      = df['nlp_index'].fillna(0)
df['nlp_index_lag1'] = df['nlp_index_lag1'].fillna(0)
print(f'Con NLP: {df.shape}')

In [ ]:
# Feature engineering para XGBoost
# XGBoost no tiene memoria temporal -> creamos lag features manualmente
TARGET = 'produccion_t'
META   = ['fecha_evento', TARGET]
EXOG   = [c for c in df.columns if c not in META]

# Lags de la variable target (lo que el LSTM aprende automaticamente)
for lag in [1, 2, 3, 6]:
    df[f'prod_lag{lag}'] = df[TARGET].shift(lag)

# Rolling statistics
df['prod_roll3_mean'] = df[TARGET].shift(1).rolling(3).mean()
df['prod_roll6_mean'] = df[TARGET].shift(1).rolling(6).mean()
df['prod_roll3_std']  = df[TARGET].shift(1).rolling(3).std()

# Eliminar filas con NaN por los lags
df_model = df.dropna().reset_index(drop=True)
print(f'Dataset con lag features: {df_model.shape}')
print(f'Filas eliminadas por NaN: {len(df) - len(df_model)}')

ALL_FEATURES = EXOG + [f'prod_lag{l}' for l in [1,2,3,6]] + ['prod_roll3_mean','prod_roll6_mean','prod_roll3_std']
print(f'Total features XGBoost: {len(ALL_FEATURES)}')
print(ALL_FEATURES)

In [ ]:
# Split cronologico 80/20
n_total = len(df_model)
n_train = int(n_total * 0.80)
n_test  = n_total - n_train

df_train = df_model.iloc[:n_train].copy()
df_test  = df_model.iloc[n_train:].copy()

print(f'Train: {n_train} | {df_train["fecha_evento"].min().date()} -> {df_train["fecha_evento"].max().date()}')
print(f'Test:  {n_test}  | {df_test["fecha_evento"].min().date()} -> {df_test["fecha_evento"].max().date()}')

X_train = df_train[ALL_FEATURES].values
y_train = df_train[TARGET].values
X_test  = df_test[ALL_FEATURES].values
y_test  = df_test[TARGET].values

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')

In [ ]:
# Busqueda de hiperparametros con TimeSeriesSplit
# Importante: shuffle=False siempre en series temporales
from itertools import product

param_grid = {
    'max_depth':        [2, 3, 4],
    'n_estimators':     [50, 100, 200],
    'learning_rate':    [0.05, 0.1, 0.2],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

tscv = TimeSeriesSplit(n_splits=3)
best_mae  = float('inf')
best_params = {}
results = []

combos = list(product(
    param_grid['max_depth'],
    param_grid['n_estimators'],
    param_grid['learning_rate'],
    param_grid['subsample'],
    param_grid['colsample_bytree']
))

print(f'Evaluando {len(combos)} combinaciones...')

for md_, ne, lr, ss, cb in combos:
    maes = []
    for tr_idx, va_idx in tscv.split(X_train):
        model_cv = xgb.XGBRegressor(
            max_depth=md_, n_estimators=ne, learning_rate=lr,
            subsample=ss, colsample_bytree=cb,
            random_state=SEED, verbosity=0
        )
        model_cv.fit(X_train[tr_idx], y_train[tr_idx])
        pred = model_cv.predict(X_train[va_idx])
        maes.append(mean_absolute_error(y_train[va_idx], pred))
    mae_cv = np.mean(maes)
    results.append({'max_depth':md_,'n_estimators':ne,'lr':lr,'ss':ss,'cb':cb,'mae_cv':mae_cv})
    if mae_cv < best_mae:
        best_mae = mae_cv
        best_params = {'max_depth':md_,'n_estimators':ne,'learning_rate':lr,'subsample':ss,'colsample_bytree':cb}

print(f'Mejor MAE CV: {best_mae:.4f}')
print(f'Mejores params: {best_params}')

In [ ]:
# Entrenar modelo final con mejores hiperparametros
model_xgb = xgb.XGBRegressor(
    **best_params,
    random_state=SEED,
    verbosity=0
)
model_xgb.fit(X_train, y_train)

y_pred = model_xgb.predict(X_test)

mae  = float(mean_absolute_error(y_test, y_pred))
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
r2   = float(r2_score(y_test, y_pred))
mape = float(np.mean(np.abs((y_test - y_pred) / (np.abs(y_test) + 1e-8))) * 100)
smape= float(100 * np.mean(2*np.abs(y_test-y_pred)/(np.abs(y_test)+np.abs(y_pred)+1e-8)))
naive_mae = mean_absolute_error(y_test, df_model[TARGET].iloc[n_train-1:-1].values if n_test > 0 else y_test)
mase = mae / (naive_mae + 1e-8)

print('=' * 55)
print('  METRICAS XGBOOST')
print('=' * 55)
print(f'  MAE:   {mae:.4f}')
print(f'  RMSE:  {rmse:.4f}')
print(f'  R2:    {r2:.4f}')
print(f'  sMAPE: {smape:.2f}%')
print(f'  MASE:  {mase:.4f}')
print('=' * 55)

In [ ]:
# Comparativa completa
if GE_METRICAS.exists():
    with open(GE_METRICAS) as f:
        ge = json.load(f)
    ge_mae  = ge.get('mae',  ge.get('MAE',  0.0673))
    ge_rmse = ge.get('rmse', ge.get('RMSE', 0.0698))
    ge_r2   = ge.get('r2',   ge.get('R2',   -2.34))
else:
    ge_mae, ge_rmse, ge_r2 = 0.0673, 0.0698, -2.34

modelos = [
    ('Naive',          0.0161, None,   None,  1.00),
    ('SARIMA',         0.1006, 0.1179, -7.86, 6.26),
    ('Prophet',        0.0919, 0.1051, -6.05, 5.72),
    ('SARIMAX+LSTM',   0.1969, 0.2926, -53.63,12.26),
    ('GE sin NLP',     ge_mae, ge_rmse, ge_r2, 4.19),
    ('GM original',    0.0981, 0.1007, -5.96, 6.11),
    ('GM v2 (NLP)',    0.0646, 0.0771, -9.86, 4.02),
    ('XGBoost',        mae,    rmse,    r2,   mase),
]

print('=' * 70)
print('  COMPARATIVA FINAL — TODOS LOS MODELOS')
print('=' * 70)
print(f"  {'Modelo':<22} {'MAE':>8} {'RMSE':>8} {'R2':>8} {'MASE':>8}")
print('-' * 70)
for nombre, m, r, r2_, ms in modelos:
    r_str  = f'{r:.4f}' if r is not None else 'N/A'
    r2_str = f'{r2_:.2f}' if r2_ is not None else 'N/A'
    ms_str = f'{ms:.4f}' if ms is not None else 'N/A'
    marca = ' <-- MEJOR MAE' if m == min(x[1] for x in modelos) else ''
    print(f"  {nombre:<22} {m:>8.4f} {r_str:>8} {r2_str:>8} {ms_str:>8}{marca}")
print('=' * 70)

if mae < 0.0646:
    print('  ALERTA: XGBoost supera al GM v2 en MAE global')
    print('  -> Verificar si en shocks el LSTM-Attention mantiene ventaja')
else:
    print('  GM v2 mantiene el mejor MAE global')
    print('  XGBoost confirma que LSTM-Attention es superior en este contexto')

In [ ]:
# Analisis de shocks para XGBoost
df_model['variacion_pct'] = df_model[TARGET].pct_change().abs() * 100
df_test_copy = df_model.iloc[n_train:].copy().reset_index(drop=True)
idx_shock = df_test_copy[df_test_copy['variacion_pct'] > 20].index.tolist()

print(f'Meses de shock en test: {len(idx_shock)} / {n_test}')

if len(idx_shock) > 0:
    mae_shock  = float(mean_absolute_error(y_test[idx_shock], y_pred[idx_shock]))
    mae_normal = float(mean_absolute_error(
        np.delete(y_test, idx_shock),
        np.delete(y_pred, idx_shock)
    )) if len(idx_shock) < n_test else mae
    deterioro = (mae_shock - mae) / mae * 100
    print(f'MAE global:  {mae:.4f}')
    print(f'MAE shocks:  {mae_shock:.4f}')
    print(f'MAE normal:  {mae_normal:.4f}')
    print(f'Deterioro en shocks: {deterioro:+.1f}%')
    print()
    print('Comparativa de deterioro en shocks:')
    print(f'  GE sin NLP: +2.3%   <- mas robusto')
    print(f'  GM v2:      +29.2%')
    print(f'  XGBoost:    {deterioro:+.1f}%')
    if deterioro < 29.2:
        print('  XGBoost mas robusto que GM v2 en shocks')
    else:
        print('  GM v2 mas robusto que XGBoost en shocks')

In [ ]:
# Feature importance XGBoost vs SHAP GE
importances = model_xgb.feature_importances_
feat_imp = pd.DataFrame({'feature': ALL_FEATURES, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_imp['feature'][::-1], feat_imp['importance'][::-1], color='steelblue', alpha=0.8)
ax.set_title('XGBoost Feature Importance (Top 15)', fontweight='bold')
ax.set_xlabel('Importancia')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(OUT_DIR / 'xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('Top 10 features:')
print(feat_imp.head(10).to_string())

In [ ]:
# Grafico predicciones vs real
fechas_test = df_test['fecha_evento'].values

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fechas_test, y_test,  'o-',  color='black',      lw=2, ms=5, label='Real')
ax.plot(fechas_test, y_pred,  's--', color='darkorange',  lw=2, ms=5, label=f'XGBoost MAE={mae:.4f}')

gm_pred_path = PROJECT_ROOT / 'resultados/gm_v2/gm_v2_predicciones.csv'
if gm_pred_path.exists():
    df_gm = pd.read_csv(gm_pred_path)
    n_ov = min(len(fechas_test), len(df_gm))
    ax.plot(fechas_test[:n_ov], df_gm['pred_gm_v2'].values[:n_ov],
            '^:', color='purple', lw=1.5, ms=5, alpha=0.7, label=f'GM v2 MAE=0.0646')

ge_pred_path = PROJECT_ROOT / 'resultados/ge/ge_predicciones.csv'
if ge_pred_path.exists():
    df_ge = pd.read_csv(ge_pred_path)
    col_pred = [c for c in df_ge.columns if 'pred' in c.lower()][0]
    n_ov = min(len(fechas_test), len(df_ge))
    ax.plot(fechas_test[:n_ov], df_ge[col_pred].values[:n_ov],
            'v:', color='steelblue', lw=1.5, ms=5, alpha=0.7, label=f'GE MAE=0.0673')

ax.set_title('XGBoost vs LSTM-Attention — Predicciones vs Real', fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Produccion (media provincial)')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUT_DIR / 'xgb_predicciones_vs_real.png', dpi=150, bbox_inches='tight')
plt.close()
print('Graficos guardados')

In [ ]:
# Guardar resultados
resultados = {
    'modelo': 'XGBoost_competidor',
    'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape, 'sMAPE': smape, 'MASE': mase,
    'n_train': n_train, 'n_test': n_test,
    'best_params': best_params,
    'n_features': len(ALL_FEATURES),
    'comparativa': {
        'Naive':        {'MAE': 0.0161, 'MASE': 1.00},
        'SARIMA':       {'MAE': 0.1006, 'MASE': 6.26},
        'Prophet':      {'MAE': 0.0919, 'MASE': 5.72},
        'SARIMAX_LSTM': {'MAE': 0.1969, 'MASE': 12.26},
        'GE_sin_NLP':   {'MAE': ge_mae, 'MASE': 4.19},
        'GM_original':  {'MAE': 0.0981, 'MASE': 6.11},
        'GM_v2':        {'MAE': 0.0646, 'MASE': 4.02},
        'XGBoost':      {'MAE': mae,    'MASE': mase}
    }
}
with open(OUT_DIR / 'xgb_metricas.json', 'w') as f:
    json.dump(resultados, f, indent=2)

pd.DataFrame({'fecha': fechas_test, 'real': y_test, 'pred_xgb': y_pred}).to_csv(
    OUT_DIR / 'xgb_predicciones.csv', index=False)

print('Archivos guardados en resultados/xgboost/')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')
print()
print('RESUMEN EJECUTIVO')
print(f'XGBoost MAE: {mae:.4f}')
print(f'GM v2   MAE: 0.0646')
print(f'GE      MAE: 0.0673')
if mae < 0.0646:
    print('RESULTADO: XGBoost supera al GM v2 -> analizar shocks')
elif mae < 0.0673:
    print('RESULTADO: XGBoost supera al GE pero no al GM v2')
else:
    print('RESULTADO: LSTM-Attention supera a XGBoost -> arquitectura validada')